<a href="https://colab.research.google.com/github/ArkanUbaidillah/BigData26_B_2411537001_ArkanUbaidillahWarman/blob/main/Praktikum3/BD_P03_Tugas_2411537001_ArkanUbaidillahWarman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Praktikum Big Data 3
## Pipeline Dua Bulan, Kualitas Komparatif, Sumber Keempat, Kamus Data, Lineage




## 0. Setup lingkungan dan fungsi dasar


In [ ]:
import sys, time, os, json, gc, shutil
import pandas as pd, numpy as np, psutil, requests
from sqlalchemy import create_engine

from google.colab import drive
drive.mount("/content/drive")

DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum3/Tugas3"
for d in (DIR_MENTAH, DIR_KURASI, DIR_SIMPAN):
    os.makedirs(d, exist_ok=True)

proses = psutil.Process(os.getpid())
catatan = []
def rss_mb():
    return proses.memory_info().rss / 1024**2
def ukur(label, fungsi):
    m0, t0 = rss_mb(), time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 2),
                     "delta_rss_mb": round(rss_mb() - m0, 1)})
    print(f"[{label}] {detik:.2f} s")
    return hasil

lineage = []
def catat_sumber(nama, asal, jumlah_baris, keterangan=""):
    lineage.append({
        "sumber": nama, "asal": asal,
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": jumlah_baris, "keterangan": keterangan,
    })
    print(f"[lineage] {nama}: {jumlah_baris:,} baris")

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan
    sementara = tujuan + ".part"
    for i in range(percobaan):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                if os.path.getsize(sementara) == 0:
                    raise IOError("berkas kosong")
                os.replace(sementara, tujuan)
                return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i)
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)
    raise RuntimeError(f"gagal mengunduh: {url}")

KOLOM = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
         "trip_distance", "PULocationID", "DOLocationID", "payment_type",
         "fare_amount", "tip_amount", "total_amount"]
KUNCI = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PULocationID", "DOLocationID", "total_amount"]
print("Setup siap. DIR_SIMPAN =", DIR_SIMPAN)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup siap. DIR_SIMPAN = /content/drive/MyDrive/BigData/Praktikum3/Tugas3


## 1. Pipeline dua bulan (Januari dan Juli 2023)

Menulis ke direktori berpartisi yang sama (`trips_bersih_dua_bulan/`) dengan partisi `bulan` + `borough_naik`.
Bukti tidak ada duplikasi antar-bulan diperiksa lewat kunci + kolom bulan.


In [ ]:
URL_ZONA = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
PATH_ZONA = os.path.join(DIR_MENTAH, "taxi_zone_lookup.csv")
unduh_aman(URL_ZONA, PATH_ZONA)
zona = pd.read_csv(PATH_ZONA)

API_CUACA = "https://archive-api.open-meteo.com/v1/archive"

def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060, percobaan=3):
    parameter = {
        "latitude": lat, "longitude": lon,
        "start_date": mulai, "end_date": selesai,
        "hourly": "temperature_2m,precipitation",
        "timezone": "America/New_York",
    }
    for i in range(percobaan):
        r = requests.get(API_CUACA, params=parameter, timeout=60)
        if r.status_code == 200:
            data = r.json()
            if "hourly" not in data:
                raise ValueError("kunci hourly hilang")
            df = pd.DataFrame(data["hourly"])
            df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)
            return df.rename(columns={"time": "jam_mulai",
                                      "temperature_2m": "suhu_c",
                                      "precipitation": "hujan_mm"})
        if r.status_code in (429, 500, 502, 503):
            time.sleep(2 ** i); continue
        r.raise_for_status()
    raise RuntimeError("API cuaca gagal")

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": ["Kartu kredit", "Tunai", "Gratis",
                        "Sengketa", "Tidak diketahui", "Perjalanan batal"],
})

def buat_laporan_kualitas(trip, zona, label_bulan):
    trip = trip.copy()
    trip["durasi_menit"] = ((trip["tpep_dropoff_datetime"] - trip["tpep_pickup_datetime"])
                            .dt.total_seconds() / 60)
    AWAL = pd.Timestamp(f"{label_bulan}-01")
    AKHIR = (AWAL + pd.offsets.MonthBegin(1))
    zona_sah = set(zona["LocationID"])
    aturan = {
        "kelengkapan: passenger_count ada": trip["passenger_count"].notna(),
        "keunikan: baris tidak duplikat": ~trip.duplicated(),
        "validitas: jarak 0-100 mil": trip["trip_distance"].between(0.01, 100),
        "validitas: total_amount > 0": trip["total_amount"] > 0,
        "validitas: PULocationID dikenal": trip["PULocationID"].isin(zona_sah),
        "konsistensi: durasi 1-180 menit": trip["durasi_menit"].between(1, 180),
        "konsistensi: total >= fare": trip["total_amount"] >= trip["fare_amount"],
        "ketepatan waktu: dalam bulan": trip["tpep_pickup_datetime"].between(AWAL, AKHIR),
    }
    lap = pd.DataFrame([
        {"bulan": label_bulan, "aturan": n,
         "lulus": int(m.sum()), "gagal": int((~m).sum()),
         "persen_gagal": round(100 * (~m).mean(), 3)}
        for n, m in aturan.items()
    ])
    return lap

def pipeline_bulan(kode_bulan):
    """kode_bulan: '2023-01' atau '2023-07'"""
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{kode_bulan}.parquet"
    path = os.path.join(DIR_MENTAH, f"yellow_tripdata_{kode_bulan}.parquet")
    ukur(f"unduh {kode_bulan}", lambda: unduh_aman(url, path))
    df = pd.read_parquet(path, columns=KOLOM)
    catat_sumber(f"trip_{kode_bulan}", url, len(df), "Parquet bulanan TLC")
    lap = buat_laporan_kualitas(df, zona, kode_bulan)

    mulai = f"{kode_bulan}-01"
    # akhir bulan
    akhir = (pd.Timestamp(mulai) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    cuaca = ambil_cuaca(mulai, akhir)
    catat_sumber(f"cuaca_{kode_bulan}", API_CUACA, len(cuaca), "Open-Meteo")

    df["durasi_menit"] = ((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
                          .dt.total_seconds() / 60)
    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(
        df.groupby("jam")["passenger_count"].transform("median"))
    df = df.drop_duplicates(subset=KUNCI)
    df = df[df["trip_distance"].between(0.01, 100)
            & df["durasi_menit"].between(1, 180)
            & (df["total_amount"] > 0)].copy()
    df = (df.merge(zona[["LocationID", "Borough"]]
                    .rename(columns={"Borough": "borough_naik"}),
                 left_on="PULocationID", right_on="LocationID",
                 how="left", validate="many_to_one")
            .drop(columns="LocationID")
            .merge(tarif_referensi, on="payment_type", how="left", validate="many_to_one"))
    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h")
    df = df.merge(cuaca, on="jam_mulai", how="left", validate="many_to_one")
    df["bulan"] = kode_bulan
    df["borough_naik"] = df["borough_naik"].fillna("Unknown").astype(str)
    print(f"{kode_bulan}: baris bersih {len(df):,}")
    return df, lap

hasil_jan, lap_jan = ukur("pipeline 2023-01", lambda: pipeline_bulan("2023-01"))
# unpack if tuple from ukur? pipeline returns tuple - handle
if isinstance(hasil_jan, tuple):
    hasil_jan, lap_jan = hasil_jan
hasil_jul, lap_jul = ukur("pipeline 2023-07", lambda: pipeline_bulan("2023-07"))
if isinstance(hasil_jul, tuple):
    hasil_jul, lap_jul = hasil_jul

gabungan = pd.concat([hasil_jan, hasil_jul], ignore_index=True)
print("total gabungan:", f"{len(gabungan):,}")

# Bukti tidak ada duplikasi antar-bulan pada kunci + bulan
n_dup = gabungan.duplicated(subset=KUNCI + ["bulan"]).sum()
print("duplikat pada kunci+bulan:", int(n_dup))
# Cek tumpang tindih identitas lintas bulan (seharusnya sangat jarang / nol untuk kunci ketat)
n_lintas = (gabungan.groupby(KUNCI)["bulan"].nunique() > 1).sum()
print("kunci yang muncul di >1 bulan:", int(n_lintas))

OUT2 = os.path.join(DIR_KURASI, "trips_bersih_dua_bulan")
if os.path.exists(OUT2):
    shutil.rmtree(OUT2)
gabungan.to_parquet(OUT2, partition_cols=["bulan", "borough_naik"], index=False)
print("partisi:", sorted(os.listdir(OUT2)))
shutil.make_archive(f"{DIR_SIMPAN}/trips_bersih_dua_bulan", "zip", OUT2)
print("zip disimpan ke Drive Praktikum3")


cache ditemukan: taxi_zone_lookup.csv
cache ditemukan: yellow_tripdata_2023-01.parquet
[unduh 2023-01] 0.00 s
[lineage] trip_2023-01: 3,066,766 baris
[lineage] cuaca_2023-01: 744 baris
2023-01: baris bersih 2,986,910
[pipeline 2023-01] 10.76 s
cache ditemukan: yellow_tripdata_2023-07.parquet
[unduh 2023-07] 0.00 s
[lineage] trip_2023-07: 2,907,108 baris
[lineage] cuaca_2023-07: 744 baris
2023-07: baris bersih 2,817,657
[pipeline 2023-07] 9.56 s
total gabungan: 5,804,567
duplikat pada kunci+bulan: 0
kunci yang muncul di >1 bulan: 0
partisi: ['bulan=2023-01', 'bulan=2023-07']
zip disimpan ke Drive Praktikum3


## 2. Laporan kualitas komparatif Januari vs Juli


In [ ]:
komparatif = pd.concat([lap_jan, lap_jul], ignore_index=True)
pivot = komparatif.pivot_table(index="aturan", columns="bulan",
                               values="persen_gagal", aggfunc="first")
pivot["selisih_pp"] = (pivot["2023-07"] - pivot["2023-01"]).round(3)
pivot = pivot.sort_values("selisih_pp", key=lambda s: s.abs(), ascending=False)
print(pivot.to_string())
pivot.to_csv(f"{DIR_SIMPAN}/laporan_kualitas_komparatif.csv")
print("tersimpan: laporan_kualitas_komparatif.csv")


bulan                             2023-01  2023-07  selisih_pp
aturan                                                        
kelengkapan: passenger_count ada    2.339    2.927       0.588
konsistensi: total >= fare          0.816    1.053       0.237
validitas: total_amount > 0         0.840    1.075       0.235
validitas: jarak 0-100 mil          1.498    1.717       0.219
konsistensi: durasi 1-180 menit     1.186    1.311       0.125
ketepatan waktu: dalam bulan        0.002    0.002       0.000
keunikan: baris tidak duplikat      0.000    0.000       0.000
validitas: PULocationID dikenal     0.000    0.000       0.000
tersimpan: laporan_kualitas_komparatif.csv


### Bacaan komparatif
Aturan dengan perubahan paling mencolok biasanya kelengkapan `passenger_count` dan validitas jarak.
Dugaan penyebab: pola liburan musim panas (Juli) mengubah komposisi perjalanan dan kelengkapan meter/penumpang, serta proporsi perjalanan bandara jarak jauh.
Angka pasti diambil dari pivot hasil eksekusi di atas.


## 3. Sumber keempat: hari libur publik (API Nager.Date, legal dan tanpa kunci)


In [ ]:
# Dokumentasi: https://date.nager.at/Api
URL_LIBUR = "https://date.nager.at/api/v3/PublicHolidays/2023/US"
r = requests.get(URL_LIBUR, timeout=30)
r.raise_for_status()
libur = pd.DataFrame(r.json())
libur["date"] = pd.to_datetime(libur["date"])
libur = libur.rename(columns={"date": "tanggal_libur", "localName": "nama_libur"})
# The original line removed `global` as it's not needed for the merge key uniqueness
libur = libur[["tanggal_libur", "nama_libur", "name", "global"]].drop_duplicates()

# Ensure `libur` has unique dates for the merge
# If there are multiple holidays on the same date, pick one (e.g., the first one encountered)
unique_libur_dates = libur.drop_duplicates(subset=["tanggal_libur"])[["tanggal_libur", "nama_libur"]]

catat_sumber("hari_libur_US", URL_LIBUR, len(libur), "Nager.Date Public Holiday API")
print(libur.head(10).to_string(index=False))
print("jumlah hari libur:", len(libur))

# Integrasi: tandai apakah tanggal pickup termasuk hari libur nasional US
gabungan = gabungan.copy()
gabungan["tanggal"] = gabungan["tpep_pickup_datetime"].dt.floor("D")
gabungan = gabungan.merge(
    unique_libur_dates.rename(columns={"tanggal_libur": "tanggal"}),
    on="tanggal", how="left", validate="many_to_one")
gabungan["is_libur"] = gabungan["nama_libur"].notna().astype("int8")
print("proporsi perjalanan pada hari libur:",
      round(100 * gabungan["is_libur"].mean(), 3), "%")
print(gabungan.groupby("is_libur")["total_amount"].median())

libur.to_csv(f"{DIR_SIMPAN}/hari_libur_US_2023.csv", index=False)

[lineage] hari_libur_US: 16 baris
tanggal_libur                           nama_libur                                 name  global
   2023-01-02                       New Year's Day                       New Year's Day    True
   2023-01-16          Martin Luther King, Jr. Day          Martin Luther King, Jr. Day    True
   2023-02-12                   Lincoln's Birthday                   Lincoln's Birthday   False
   2023-02-20                Washington's Birthday                       Presidents Day    True
   2023-04-07                          Good Friday                          Good Friday   False
   2023-05-08                           Truman Day                           Truman Day   False
   2023-05-29                         Memorial Day                         Memorial Day    True
   2023-06-19 Juneteenth National Independence Day Juneteenth National Independence Day    True
   2023-07-04                     Independence Day                     Independence Day    True
   202

## 4. Kamus data (`kamus_data.md`)


In [ ]:
kamus = '''# Kamus Data — Dataset Akhir Praktikum 3
# trips_bersih (hasil pipeline)

| Kolom | Tipe | Satuan | Sumber | Aturan validitas | Penanganan nilai hilang |
|---|---|---|---|---|---|
| tpep_pickup_datetime | datetime64[ns] | waktu lokal NYC | TLC Yellow Taxi Parquet | harus dalam bulan file | baris di luar periode ditandai gagal ketepatan waktu |
| tpep_dropoff_datetime | datetime64[ns] | waktu lokal NYC | TLC | dropoff >= pickup | baris inkonsisten dibuang pada filter durasi |
| passenger_count | float/int | orang | TLC | biasanya 1-6 | MAR: isi median per jam + flag passenger_count_hilang |
| passenger_count_hilang | int8 | flag 0/1 | turunan | 0 atau 1 | tidak hilang (turunan) |
| trip_distance | float64 | mil | TLC | 0.01-100 | di luar rentang dibuang |
| trip_distance_capped | float64 | mil | turunan | <= p99.5 | winsorize upper |
| PULocationID | int | ID zona | TLC | harus ada di taxi_zone_lookup | left join; NA borough = Unknown |
| DOLocationID | int | ID zona | TLC | harus ada di lookup | sama |
| payment_type | int | kode 1-6 | TLC | 1-6 | join ke tarif_referensi |
| fare_amount | float64 | USD | TLC | biasanya >= 0 | dipakai cek total >= fare |
| tip_amount | float64 | USD | TLC | >= 0 | tidak diisi ulang |
| total_amount | float64 | USD | TLC | > 0 | <= 0 dibuang; outlier ditandai tarif_ekstrem |
| durasi_menit | float64 | menit | turunan (dropoff-pickup) | 1-180 | di luar rentang dibuang |
| jam | int | 0-23 | turunan | 0-23 | dari pickup |
| jam_mulai | datetime64[ns] | jam | turunan floor h | - | kunci join cuaca |
| borough_naik | object | nama borough | taxi_zone_lookup | daftar borough NYC | NA -> Unknown |
| zona_naik | object | nama zona | taxi_zone_lookup | - | NA bila LocationID tak dikenal |
| nama_pembayaran | object | label | SQLite tarif_referensi | 6 label | left join |
| suhu_c | float64 | derajat C | Open-Meteo | wajar untuk NYC musim terkait | biarkan NA (sedikit) |
| hujan_mm | float64 | mm/jam | Open-Meteo | >= 0 | fillna 0 saat buat flag hujan |
| hujan | int8 | flag 0/1 | turunan | 0/1 | hujan_mm > 0.1 |
| tarif_ekstrem | int8 | flag 0/1 | turunan IQR | 0/1 | penanda, nilai tidak diubah |
| hari_minggu | int | 0-6 | turunan | 0-6 | - |
| akhir_pekan | int8 | flag | turunan | 0/1 | - |
| kecepatan_mph | float64 | mil/jam | turunan | > 0 | jarak / (durasi/60) |
| tarif_per_mil | float64 | USD/mil | turunan | > 0 | total / jarak |
| tanggal | object/date | tanggal | turunan | - | partisi bantu |
| bulan | object | YYYY-MM | parameter pipeline | - | partisi |
| is_libur | int8 | flag | Nager.Date | 0/1 | left join hari libur |
| nama_libur | object | teks | Nager.Date | - | NA bila bukan libur |

## Catatan etis/legal
- TLC Trip Record: data publik pemerintah NYC, boleh dipakai penelitian/pendidikan.
- Open-Meteo: API publik, tanpa kunci, patuhi batas wajar permintaan.
- Nager.Date: API publik hari libur, tanpa kunci.
- Jangan menimpa lapisan_mentah/. Semua olahan ke lapisan_terkurasi/ atau Drive Praktikum3.
'''
path_kamus = f"{DIR_SIMPAN}/kamus_data.md"
with open(path_kamus, "w") as f:
    f.write(kamus)
print("kamus disimpan:", path_kamus)
print(kamus[:500], "...")


kamus disimpan: /content/drive/MyDrive/BigData/Praktikum3/Tugas3/kamus_data.md
# Kamus Data — Dataset Akhir Praktikum 3
# trips_bersih (hasil pipeline)

| Kolom | Tipe | Satuan | Sumber | Aturan validitas | Penanganan nilai hilang |
|---|---|---|---|---|---|
| tpep_pickup_datetime | datetime64[ns] | waktu lokal NYC | TLC Yellow Taxi Parquet | harus dalam bulan file | baris di luar periode ditandai gagal ketepatan waktu |
| tpep_dropoff_datetime | datetime64[ns] | waktu lokal NYC | TLC | dropoff >= pickup | baris inkonsisten dibuang pada filter durasi |
| passenger_count |  ...


## 5. Dokumen lineage


In [ ]:
# Simpan lineage terstruktur
pd.DataFrame(lineage).to_csv(f"{DIR_SIMPAN}/lineage.csv", index=False)
print(pd.DataFrame(lineage).to_string(index=False))

# Diagram alur tekstual (satu halaman)
alur = '''
DATA LINEAGE — PRAKTIKUM 3 / TUGAS
=================================

[1] TLC yellow_tripdata_YYYY-MM.parquet  --unduh_aman-->  lapisan_mentah/
[2] TLC taxi_zone_lookup.csv             --unduh_aman-->  lapisan_mentah/
[3] Open-Meteo Archive API (hourly)      --ambil_cuaca-->  DataFrame cuaca
[4] SQLite operasional.db (tarif_referensi) --to_sql/read_sql-->  tabel dimensi
[5] Nager.Date Public Holidays US 2023   --requests-->  hari_libur

        (1) trip mentah
              |
              v
        hitung durasi_menit, jam
              |
              v
        isi passenger_count (median per jam) + flag _hilang
              |
              v
        drop_duplicates(KUNCI)           ## pembuangan: duplikat kunci
              |
              v
        filter bisnis (jarak, durasi, total>0)  ## pembuangan: anomali rentang
              |
              +---- left join (2) zona  validate=many_to_one
              +---- left join (4) tarif validate=many_to_one
              +---- left join (3) cuaca on jam_mulai
              +---- left join (5) libur on tanggal   (tugas)
              |
              v
        validasi(KONTRAK)  ## gagal berisik bila kontrak dilanggar
              |
              v
        Parquet berpartisi  lapisan_terkurasi/trips_bersih[_dua_bulan]/
              |
              v
        Artefak Drive: laporan_kualitas*.csv, lineage.csv,
                       pengukuran_kinerja.csv, trips_bersih*.zip, kamus_data.md
'''
print(alur)
with open(f"{DIR_SIMPAN}/lineage_diagram.txt", "w") as f:
    f.write(alur)


       sumber                                                                            asal                     diambil_pada  jumlah_baris                    keterangan
 trip_2023-01 https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet 2026-09-25T02:41:30.097441+00:00       3066766           Parquet bulanan TLC
cuaca_2023-01                                   https://archive-api.open-meteo.com/v1/archive 2026-09-25T02:41:34.147159+00:00           744                    Open-Meteo
 trip_2023-07 https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-07.parquet 2026-09-25T02:41:39.382935+00:00       2907108           Parquet bulanan TLC
cuaca_2023-07                                   https://archive-api.open-meteo.com/v1/archive 2026-09-25T02:41:42.654119+00:00           744                    Open-Meteo
hari_libur_US                             https://date.nager.at/api/v3/PublicHolidays/2023/US 2026-09-25T02:42:24.577385+00:00            16 Nage